# Historical Backtesting

This notebook evaluates the quantitative screener developed in
`02_quant_screener.ipynb` using historical data.

The objective is to determine whether the asset-selection methodology and
portfolio-weighting rules would have produced robust results historically.

At each monthly rebalance date, the strategy:

1. Uses only information available up to that date.
2. Builds the eligible investment universe.
3. Calculates the cross-sectional factor score.
4. Selects the highest-ranked assets within each category.
5. Assigns portfolio weights.
6. Enters positions on the first available observation after the signal date.
7. Holds the portfolio until the following rebalance date.

The analysis separates:

$$
\text{Calibration: 2021--2024}
$$

from:

$$
\text{Out-of-sample: 2025--2026}.
$$

This separation reduces the risk of selecting parameters solely because they
performed well over the complete historical sample.

In [1]:
# Imports

from pathlib import Path
import sys


import numpy as np
import pandas as pd

from sqlalchemy import text


# Project root

PROJECT_ROOT = Path.cwd()

while not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT)
    )


# Project modules

from src.database import get_engine
from src.features import calculate_features
from src.screener import (
    build_universe,
    select_candidates,
    select_candidates_with_buffer
)
from src.portfolio import calculate_portfolio_weights

In [2]:
# Database connection and prices

engine = get_engine()

with engine.connect() as conn:
    conn.execute(text("SELECT 1"))

print("Database connection successful.")


prices = pd.read_sql(
    """
    SELECT
        p.date,
        p.asset_id,
        a.actinver_symbol,
        a.asset_type,
        p.adjusted_close,
        p.volume

    FROM public.prices p

    JOIN public.assets a
        ON p.asset_id = a.asset_id

    ORDER BY
        p.asset_id,
        p.date;
    """,
    engine
)

prices["date"] = pd.to_datetime(
    prices["date"]
)

print(f"Rows: {len(prices):,}")
print(
    f"Assets: "
    f"{prices['asset_id'].nunique()}"
)
print(
    f"Period: "
    f"{prices['date'].min().date()} "
    f"to {prices['date'].max().date()}"
)

Database connection successful.
Rows: 342,360
Assets: 217
Period: 2020-01-02 to 2026-10-02


## 1. Historical features

The same feature-engineering methodology used by the current screener is used
throughout the backtest.

The model includes:

- 5-observation momentum
- 20-observation momentum
- 60-observation momentum
- 20-observation annualized volatility
- 252-observation current drawdown
- recent trading-activity indicators
- recent extreme-return filters

Using the same functions in both the current screener and historical backtest
ensures methodological consistency.

In [3]:
# Feature calculation

prices = calculate_features(
    prices
)

In [4]:
# Backtest configuration

BACKTEST_START = pd.Timestamp(
    "2021-01-01"
)

CALIBRATION_END = pd.Timestamp(
    "2024-12-31"
)

OOS_START = pd.Timestamp(
    "2025-01-01"
)

ENTRY_CUTOFF = 0.90


SCORE_MODELS = {
    "Baseline": {
        "momentum": 1 / 3,
        "volatility": 1 / 3,
        "drawdown": 1 / 3
    },

    "Defensive": {
        "momentum": 0.30,
        "volatility": 0.45,
        "drawdown": 0.25
    }
}


WEIGHTING_METHODS = [
    "equal",
    "category_inverse_volatility",
    "category_score_volatility"
]

In [5]:
# Rebalance dates

market_dates = (
    prices["date"]
    .drop_duplicates()
    .sort_values()
)

rebalance_dates = (
    market_dates
    .groupby(
        market_dates.dt.to_period("M")
    )
    .max()
)

rebalance_dates = pd.DatetimeIndex(
    rebalance_dates
)

rebalance_dates = rebalance_dates[
    rebalance_dates >= BACKTEST_START
]

print(
    f"Rebalance dates: "
    f"{len(rebalance_dates)}"
)

print(
    f"First: "
    f"{rebalance_dates.min().date()}"
)

print(
    f"Last: "
    f"{rebalance_dates.max().date()}"
)

Rebalance dates: 70
First: 2021-01-29
Last: 2026-10-02


## 2. Backtest rules

The strategy is rebalanced monthly.

At the end of each month $t$, the model calculates the screener using only
information available through that date.

The portfolio is then entered using the first available adjusted closing price
after the signal date.

For asset $i$:

$$
R_{i,t}
=
\frac{P_{i,t+1}^{exit}}
     {P_{i,t+1}^{entry}}
-1.
$$

Portfolio return is:

$$
R_{p,t}
=
\sum_{i=1}^{N}
w_{i,t}R_{i,t}.
$$

This timing convention prevents the model from using the same closing price
both to generate a signal and to assume execution retroactively.

In [6]:
# Period return

def calculate_period_return(
    prices,
    candidates,
    start_date,
    end_date
):

    if candidates.empty:
        return np.nan

    selected_ids = (
        candidates["asset_id"]
        .unique()
    )

    period = prices[
        prices["asset_id"].isin(
            selected_ids
        )
        & (
            prices["date"]
            > start_date
        )
        & (
            prices["date"]
            <= end_date
        )
    ].copy()

    if period.empty:
        return np.nan

    period = period.sort_values(
        ["asset_id", "date"]
    )

    entry_prices = (
        period
        .groupby("asset_id")
        .first()["adjusted_close"]
    )

    exit_prices = (
        period
        .groupby("asset_id")
        .last()["adjusted_close"]
    )

    asset_returns = (
        exit_prices
        / entry_prices
        - 1
    )

    return_data = (
        candidates[
            [
                "asset_id",
                "weight"
            ]
        ]
        .drop_duplicates("asset_id")
        .set_index("asset_id")
        .join(
            asset_returns.rename(
                "asset_return"
            )
        )
        .replace(
            [np.inf, -np.inf],
            np.nan
        )
        .dropna()
    )

    if return_data.empty:
        return np.nan

    # Renormalize if an asset lacks a valid return.
    return_data["weight"] = (
        return_data["weight"]
        / return_data["weight"].sum()
    )

    portfolio_return = (
        return_data["weight"]
        * return_data["asset_return"]
    ).sum()

    return portfolio_return

In [7]:
# Main backtest function

def run_backtest(
    prices,
    rebalance_dates,
    score_weights,
    weighting_method,
    entry_cutoff=0.90,
    use_buffer=False,
    exit_cutoff=0.80
):

    results = []
    weights_history = []

    previous_assets = set()

    for i in range(
        len(rebalance_dates) - 1
    ):

        signal_date = (
            rebalance_dates[i]
        )

        end_date = (
            rebalance_dates[i + 1]
        )

        universe = build_universe(
            prices=prices,
            evaluation_date=signal_date,
            momentum_weight=score_weights[
                "momentum"
            ],
            volatility_weight=score_weights[
                "volatility"
            ],
            drawdown_weight=score_weights[
                "drawdown"
            ]
        )

        if use_buffer:

            candidates = (
                select_candidates_with_buffer(
                    universe=universe,
                    previous_assets=previous_assets,
                    entry_cutoff=entry_cutoff,
                    exit_cutoff=exit_cutoff
                )
            )

        else:

            candidates = select_candidates(
                universe,
                cutoff=entry_cutoff
            )

        candidates = (
            calculate_portfolio_weights(
                candidates,
                method=weighting_method
            )
        )

        period_return = (
            calculate_period_return(
                prices=prices,
                candidates=candidates,
                start_date=signal_date,
                end_date=end_date
            )
        )

        results.append(
            {
                "signal_date":
                    signal_date,

                "end_date":
                    end_date,

                "n_assets":
                    len(candidates),

                "return":
                    period_return
            }
        )

        weights = candidates[
            [
                "asset_id",
                "actinver_symbol",
                "weight"
            ]
        ].copy()

        weights["signal_date"] = (
            signal_date
        )

        weights_history.append(
            weights
        )

        previous_assets = set(
            candidates["asset_id"]
        )

    backtest = pd.DataFrame(
        results
    )

    weights_history = pd.concat(
        weights_history,
        ignore_index=True
    )

    return (
        backtest,
        weights_history
    )

In [8]:
# Metrics

def calculate_metrics(
    backtest
):

    data = (
        backtest
        .dropna(
            subset=["return"]
        )
        .copy()
    )

    if data.empty:
        return {
            "Final Wealth": np.nan,
            "CAGR": np.nan,
            "Annualized Volatility": np.nan,
            "Sharpe Ratio": np.nan,
            "Maximum Drawdown": np.nan
        }

    wealth = (
        1 + data["return"]
    ).cumprod()

    final_wealth = (
        wealth.iloc[-1]
    )

    n_months = len(data)

    years = (
        n_months / 12
    )

    cagr = (
        final_wealth
        ** (1 / years)
        - 1
    )

    return_std = (
        data["return"].std()
    )

    annualized_volatility = (
        return_std
        * np.sqrt(12)
    )

    if (
        pd.isna(return_std)
        or np.isclose(return_std, 0)
    ):
        sharpe = np.nan

    else:
        sharpe = (
            data["return"].mean()
            / return_std
            * np.sqrt(12)
        )

    wealth_with_initial = pd.concat(
        [
            pd.Series([1.0]),
            wealth.reset_index(drop=True)
        ],
        ignore_index=True
    )

    running_max = (
        wealth_with_initial.cummax()
    )

    drawdown = (
        wealth_with_initial
        / running_max
        - 1
    )

    maximum_drawdown = (
        drawdown.min()
    )

    return {
        "Final Wealth":
            final_wealth,

        "CAGR":
            cagr,

        "Annualized Volatility":
            annualized_volatility,

        "Sharpe Ratio":
            sharpe,

        "Maximum Drawdown":
            maximum_drawdown
    }

In [9]:
# Turnover

def calculate_turnover(
    weights_history
):

    weight_matrix = (
        weights_history
        .pivot(
            index="signal_date",
            columns="asset_id",
            values="weight"
        )
        .fillna(0)
        .sort_index()
    )

    turnover = (
        weight_matrix
        .diff()
        .abs()
        .sum(axis=1)
        / 2
    )

    return turnover.dropna()

## 3. Score-model calibration

Two economically motivated factor specifications are evaluated.

### Baseline

$$
Score_i
=
\frac13 Momentum_i
-
\frac13 Volatility_i
+
\frac13 Drawdown_i.
$$

### Defensive

$$
Score_i
=
0.30\,Momentum_i
-
0.45\,Volatility_i
+
0.25\,Drawdown_i.
$$

The 2021--2024 period is used for calibration, while 2025--2026 is reserved
for out-of-sample evaluation.

The objective is not to maximize historical performance over a large parameter
grid, but to compare a small number of economically interpretable
specifications.

In [10]:
# Run main strategy combinations

backtests = {}
weight_histories = {}


for model_name, score_weights in (
    SCORE_MODELS.items()
):

    for weighting_method in (
        WEIGHTING_METHODS
    ):

        strategy_name = (
            f"{model_name} - "
            f"{weighting_method}"
        )

        bt, weights = run_backtest(
            prices=prices,
            rebalance_dates=rebalance_dates,
            score_weights=score_weights,
            weighting_method=weighting_method,
            entry_cutoff=ENTRY_CUTOFF,
            use_buffer=False
        )

        # Remove incomplete current month.
        current_period = (
            prices["date"]
            .max()
            .to_period("M")
        )

        bt = bt[
            bt["end_date"]
            .dt.to_period("M")
            != current_period
        ].copy()

        backtests[
            strategy_name
        ] = bt

        weight_histories[
            strategy_name
        ] = weights

In [11]:
# Calibration vs OOS

comparison_results = []


for strategy_name, bt in (
    backtests.items()
):

    periods = {
        "Calibration 2021-2024":
            bt[
                bt["end_date"]
                <= CALIBRATION_END
            ],

        "Out-of-sample 2025-2026":
            bt[
                bt["signal_date"]
                >= OOS_START
            ]
    }

    for period_name, period_bt in (
        periods.items()
    ):

        metrics = calculate_metrics(
            period_bt
        )

        metrics["Period"] = (
            period_name
        )

        metrics["Strategy"] = (
            strategy_name
        )

        comparison_results.append(
            metrics
        )


strategy_comparison = (
    pd.DataFrame(
        comparison_results
    )
    .set_index(
        [
            "Period",
            "Strategy"
        ]
    )
)


display(
    strategy_comparison[
        [
            "Final Wealth",
            "CAGR",
            "Annualized Volatility",
            "Sharpe Ratio",
            "Maximum Drawdown"
        ]
    ]
)

,,Final Wealth,CAGR,Annualized Volatility,Sharpe Ratio,Maximum Drawdown
Period,Strategy,,,,,
Calibration 2021-2024,Baseline - equal,1.560736,0.120368,0.100086,1.189047,-0.134532
Out-of-sample 2025-2026,Baseline - equal,1.057909,0.034353,0.093674,0.405244,-0.086902
Calibration 2021-2024,Baseline - category_inverse_volatility,1.553798,0.119094,0.086289,1.351709,-0.101125
Out-of-sample 2025-2026,Baseline - category_inverse_volatility,1.044573,0.026510,0.074198,0.388242,-0.079962
Calibration 2021-2024,Baseline - category_score_volatility,1.535139,0.115648,0.086307,1.315401,-0.098580
Out-of-sample 2025-2026,Baseline - category_score_volatility,1.048677,0.028928,0.075938,0.412027,-0.079442
Calibration 2021-2024,Defensive - equal,1.516197,0.112117,0.084978,1.296919,-0.116689
Out-of-sample 2025-2026,Defensive - equal,1.047068,0.027981,0.076875,0.395736,-0.077510
Calibration 2021-2024,Defensive - category_inverse_volatility,1.507857,0.110552,0.076523,1.413204,-0.095474


In [12]:
# Tunrover comparison

turnover_results = []


for strategy_name, weights in (
    weight_histories.items()
):

    turnover = calculate_turnover(
        weights
    )

    calibration_turnover = turnover[
        turnover.index
        <= CALIBRATION_END
    ]

    oos_turnover = turnover[
        turnover.index
        >= OOS_START
    ]

    turnover_results.append(
        {
            "Strategy":
                strategy_name,

            "Calibration Turnover":
                calibration_turnover.mean(),

            "OOS Turnover":
                oos_turnover.mean()
        }
    )


turnover_comparison = (
    pd.DataFrame(
        turnover_results
    )
    .set_index("Strategy")
)


turnover_comparison

,Calibration Turnover,OOS Turnover
Strategy,,
Baseline - equal,0.657810,0.719955
Baseline - category_inverse_volatility,0.689180,0.743728
Baseline - category_score_volatility,0.691455,0.746450
Defensive - equal,0.627933,0.694671
Defensive - category_inverse_volatility,0.656211,0.715981
Defensive - category_score_volatility,0.659263,0.719987


## 4. Selected strategy specification

The calibration and out-of-sample results favor a more defensive score over
momentum-heavy alternatives.

The selected V1 score is:

$$
Score_i
=
0.30\,Momentum_i
-
0.45\,Volatility_i
+
0.25\,Drawdown_i
$$

Assets are selected when:

$$
CategoryPercentile_i \geq 0.90.
$$

The preferred portfolio-weighting method is
`category score/volatility`.

For category $c$, the category budget is proportional to the number of
selected assets in that category:

$$
W_c
=
\frac{N_c}{N}.
$$

Within each category:

$$
\tilde w_{i|c}
=
\frac{\max(Score_i,0)}
{\sigma_i},
$$

and:

$$
w_{i|c}
=
\frac{\tilde w_{i|c}}
{\sum_{j\in c}\tilde w_{j|c}}.
$$

The final asset weight is:

$$
w_i
=
W_c w_{i|c}.
$$

In [13]:
# Buffer experiment

defensive_weights = (
    SCORE_MODELS["Defensive"]
)


buffer_configs = {
    "90/90 No Buffer": {
        "use_buffer": False,
        "exit_cutoff": 0.90
    },

    "90/85 Buffer": {
        "use_buffer": True,
        "exit_cutoff": 0.85
    },

    "90/80 Buffer": {
        "use_buffer": True,
        "exit_cutoff": 0.80
    }
}


buffer_backtests = {}
buffer_weights = {}


for name, config in (
    buffer_configs.items()
):

    bt, weights = run_backtest(
        prices=prices,
        rebalance_dates=rebalance_dates,
        score_weights=defensive_weights,
        weighting_method=
            "category_score_volatility",
        entry_cutoff=0.90,
        use_buffer=config[
            "use_buffer"
        ],
        exit_cutoff=config[
            "exit_cutoff"
        ]
    )

    current_period = (
        prices["date"]
        .max()
        .to_period("M")
    )

    bt = bt[
        bt["end_date"]
        .dt.to_period("M")
        != current_period
    ].copy()

    buffer_backtests[name] = bt
    buffer_weights[name] = weights

In [14]:
# Buffer OOS comparison

buffer_results = []


for name, bt in (
    buffer_backtests.items()
):

    oos_bt = bt[
        bt["signal_date"]
        >= OOS_START
    ]

    metrics = calculate_metrics(
        oos_bt
    )

    turnover = calculate_turnover(
        buffer_weights[name]
    )

    oos_turnover = turnover[
        turnover.index
        >= OOS_START
    ]

    metrics["Strategy"] = name
    metrics["OOS Turnover"] = (
        oos_turnover.mean()
    )

    buffer_results.append(
        metrics
    )


buffer_comparison = (
    pd.DataFrame(
        buffer_results
    )
    .set_index("Strategy")
)


buffer_comparison[
    [
        "CAGR",
        "Annualized Volatility",
        "Sharpe Ratio",
        "Maximum Drawdown",
        "OOS Turnover"
    ]
]

,CAGR,Annualized Volatility,Sharpe Ratio,Maximum Drawdown,OOS Turnover
Strategy,,,,,
90/90 No Buffer,0.028163,0.071322,0.423570,-0.070625,0.719987
90/85 Buffer,0.023944,0.069536,0.373541,-0.074652,0.613759
90/80 Buffer,0.020713,0.066842,0.338698,-0.078427,0.536146


### Buffer conclusion

Entry/exit buffers reduce turnover substantially, but the reduction comes at
the cost of weaker out-of-sample performance.

The no-buffer specification produced the strongest out-of-sample CAGR and
Sharpe ratio among the tested alternatives.

For this reason, the V1 strategy retains the strict 90th-percentile selection
rule despite its relatively high turnover.

High turnover remains an explicit limitation of the strategy.

In [15]:
# Transaction-cost function

def apply_turnover_costs(
    backtest,
    weights_history,
    cost_rate
):

    data = backtest.copy()

    turnover = calculate_turnover(
        weights_history
    )

    data = data.set_index(
        "signal_date"
    )

    data["turnover"] = (
        turnover
    )

    data["turnover"] = (
        data["turnover"]
        .fillna(0)
    )

    data["net_return"] = (
        data["return"]
        - cost_rate
        * data["turnover"]
    )

    return data.reset_index()

In [16]:
# Cost sensitivity

FINAL_STRATEGY = (
    "Defensive - "
    "category_score_volatility"
)

final_bt = (
    backtests[
        FINAL_STRATEGY
    ]
)

final_weights = (
    weight_histories[
        FINAL_STRATEGY
    ]
)


cost_scenarios = {
    "Gross": 0.0,
    "10 bps": 0.001,
    "25 bps": 0.0025
}


cost_results = []


for scenario, cost_rate in (
    cost_scenarios.items()
):

    net_bt = apply_turnover_costs(
        final_bt,
        final_weights,
        cost_rate
    )

    net_bt = net_bt.rename(
        columns={
            "return":
                "gross_return",

            "net_return":
                "return"
        }
    )

    metrics = calculate_metrics(
        net_bt
    )

    metrics["Scenario"] = scenario

    cost_results.append(
        metrics
    )


cost_comparison = (
    pd.DataFrame(
        cost_results
    )
    .set_index("Scenario")
)


cost_comparison[
    [
        "Final Wealth",
        "CAGR",
        "Annualized Volatility",
        "Sharpe Ratio",
        "Maximum Drawdown"
    ]
]

,Final Wealth,CAGR,Annualized Volatility,Sharpe Ratio,Maximum Drawdown
Scenario,,,,,
Gross,1.637705,0.090954,0.075800,1.189550,-0.090016
10 bps,1.564377,0.082170,0.075787,1.082373,-0.093379
25 bps,1.460407,0.069116,0.075770,0.921476,-0.098406


## 5. Conclusions

The historical analysis suggests that the quantitative screener is primarily a
risk-adjusted asset-selection strategy rather than a pure return-maximization
strategy.

The principal findings are:

- Increasing the momentum weight did not improve out-of-sample efficiency.
- A more defensive score produced more stable calibration and out-of-sample
  results.
- Global inverse-volatility weighting caused excessive allocation to
  low-volatility funds and was therefore rejected.
- Applying risk-adjusted weights within each asset category produced more
  economically meaningful allocations.
- Entry/exit buffers reduced turnover but weakened out-of-sample performance.
- Transaction costs materially reduce performance because monthly turnover
  remains high.

The selected V1 specification is:

$$
Score_i
=
0.30\,Momentum_i
-
0.45\,Volatility_i
+
0.25\,Drawdown_i
$$

with:

$$
CategoryPercentile_i \geq 0.90
$$

and category-based score/volatility weighting.

The model should therefore be interpreted as a first systematic portfolio
construction framework rather than as an optimized trading strategy.

Future improvements may include:

- explicit portfolio optimization;
- covariance-aware risk allocation;
- GARCH volatility forecasts;
- multivariate Monte Carlo simulation;
- transaction-cost-aware optimization;
- more robust walk-forward validation.